# AutoTimes — Benchmark no Kaggle (NVIDIA CUDA)

Execute todas as células em ordem.

**Modos de execução:**
- **Otimizações** (opt1~opt13 + base) — roda os scripts `run_opt*.py`
- **Bateria completa** — roda `rodar_bateria.ps1` adaptado: múltiplos datasets, modelos e tarefas

---
## 1. Instalação de dependências

In [ ]:
!pip install -q einops==0.4.0 pandas==1.5.3 scikit-learn==1.2.2 scipy==1.10.1 \
    tqdm==4.64.1 transformers==4.35.2 accelerate sentencepiece protobuf \
    reformer-pytorch==1.4.4 patool==1.12 sympy==1.11.1

In [ ]:
import sys, os, json, time, random, gc, csv, shutil
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
import subprocess

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

---
## 2. Extrair e preparar dataset

In [ ]:
# Caminho do dataset no Kaggle (faça upload como dataset separado)
KAGGLE_DATASET = "/kaggle/input/datasets/emanuellopessilva69/datasetoficial"

# Copia tudo para o working dir (substitui o dataset/ do zip)
DST = "/kaggle/working/dataset"
os.makedirs(DST, exist_ok=True)

if os.path.isdir(KAGGLE_DATASET):
    print(f"Copiando de {KAGGLE_DATASET} para {DST}...")
    for item in os.listdir(KAGGLE_DATASET):
        src = os.path.join(KAGGLE_DATASET, item)
        dst = os.path.join(DST, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print("Dataset copiado.")
    !ls -la {DST}/
else:
    print(f"AVISO: {KAGGLE_DATASET} não encontrado. Usando dataset/ do zip.")
    DST = "./dataset"

---
## 3. Configuração

Edite as variáveis abaixo para escolher o modo e parâmetros.

In [ ]:
# ========== MODO DE EXECUÇÃO ==========
# "otimizacoes" — roda os scripts run_opt*.py
# "bateria"     — roda múltiplos modelos/datasets (como rodar_bateria.ps1)
MODO = "otimizacoes"

# ========== CONFIG GERAL ==========
# Dataset base (assume que o dataset foi copiado para DST)
ROOT_PATH = DST + "/ETT-small"
DATA = "ETTh1"
DATA_PATH = "ETTh1.csv"

# Hiperparâmetros
EPOCHS = 10
BATCH_SIZE = 256
SEQ_LEN = 672
LABEL_LEN = 576
TOKEN_LEN = 96
TEST_PRED_LEN = 96
SMOKE_TEST = 0  # 1 = 1 batch/época só para testar

# ========== CONFIG MODO OTIMIZAÇÕES ==========
OPTS = [
    "base",
    "opt4", "opt5", "opt6", "opt7",
    "opt8", "opt9", "opt10", "opt11",
    "opt12", "opt13",
]

# ========== CONFIG MODO BATERIA ==========
# Modelos disponíveis: AutoTimes_Gpt2, AutoTimes_Llama, AutoTimes_Opt_1b, AutoTimes_Qwen
# Diretório do LLM: gpt2, llama, opt, qwen (baixado do HF ou path local)
BATERIA_MODELOS = ["AutoTimes_Gpt2"]
BATERIA_LLM_DIRS = ["gpt2"]

# Tarefa: long_term_forecast, short_term_forecast, zero_shot_forecast, in_context_forecast
BATERIA_TASK = "long_term_forecast"

# Datasets para bateria (lista de dicts)
BATERIA_DATASETS = [
    {"nome": "ETTh1",      "root": DST + "/ETT-small",    "path": "ETTh1.csv",      "type": "ETTh1",  "lr": "0.001"},
    # {"nome": "Electricity", "root": DST + "/electricity", "path": "electricity.csv", "type": "custom", "lr": "0.001"},
    # {"nome": "Weather",     "root": DST + "/weather",     "path": "weather.csv",     "type": "custom", "lr": "0.001"},
    # {"nome": "Traffic",     "root": DST + "/traffic",     "path": "traffic.csv",     "type": "custom", "lr": "0.001"},
]

# ========== FIM CONFIG ==========

In [ ]:
OPT_MAP = {
    "base": "run_base.py",
    "opt1": "run_opt1.py", "opt2": "run_opt2.py", "opt3": "run_opt3.py",
    "opt4": "run_opt4.py", "opt5": "run_opt5.py", "opt6": "run_opt6.py",
    "opt7": "run_opt7.py", "opt8": "run_opt8.py", "opt9": "run_opt9.py",
    "opt10": "run_opt10.py", "opt11": "run_opt11.py",
    "opt12": "run_opt12.py", "opt13": "run_opt13.py",
}

OPT_NAMES = {
    "base": "Base_Original",
    "opt1": "Opt_1_Rocm", "opt2": "Opt_2_MLA", "opt3": "Opt_3_MoE",
    "opt4": "Opt_4_GRPO", "opt5": "Opt_5_RAG_Temporal",
    "opt6": "Opt_6_Mamba_SSM", "opt7": "Opt_7_MTP",
    "opt8": "Opt_8_FinGPT", "opt9": "Opt_9_SMETimes",
    "opt10": "Opt_10_Combo_Eficiencia", "opt11": "Opt_11_Combo_Preditor",
    "opt12": "Opt_12_Ultimate", "opt13": "Opt_13_Combo_SLM_Contextual",
}

---
## 4. Funções auxiliares

In [ ]:
def cleanup_gpu():
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

def build_args(script, opt_id, opt_name, root_path, data, data_path, model="AutoTimes_Gpt2", llm_dir="gpt2"):
    args = [
        script,
        "--task_name=long_term_forecast", "--is_training=1",
        "--model_id=test", f"--model={model}",
        f"--data={data}", f"--root_path={root_path}",
        f"--data_path={data_path}", f"--llm_ckp_dir={llm_dir}",
        f"--batch_size={BATCH_SIZE}", f"--seq_len={SEQ_LEN}",
        f"--label_len={LABEL_LEN}", f"--token_len={TOKEN_LEN}",
        f"--test_pred_len={TEST_PRED_LEN}", "--use_amp",
        f"--opt_id={opt_id}", f"--opt_name={opt_name}",
    ]
    if SMOKE_TEST:
        args += ["--train_epochs=1", "--smoke_test=1"]
    else:
        args += [f"--train_epochs={EPOCHS}"]
    return args

def run_script(args, label):
    os.makedirs("logs", exist_ok=True)
    output_log = f"logs/{label}.txt"
    start = time.time()
    print(f"[EXEC] python {' '.join(args)}")
    with open(output_log, "w") as log:
        result = subprocess.run([sys.executable] + args, stdout=log, stderr=subprocess.STDOUT)
    elapsed = time.time() - start
    if result.returncode == 0:
        print(f"[OK] {label} concluído em {elapsed/60:.2f} min")
        with open(output_log) as f:
            for line in f.readlines()[-5:]:
                print(f"  {line.rstrip()}")
    else:
        print(f"[FAIL] {label} — exit {result.returncode} ({elapsed:.1f}s)")
        with open(output_log) as f:
            print(f.read())
    cleanup_gpu()
    return result.returncode

---
## 5. Execução

In [ ]:
print("="*50)
print(f"  Modo: {MODO}")
print(f"  Dataset: {ROOT_PATH}/{DATA_PATH}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print("="*50)

In [ ]:
errors = []

if MODO == "otimizacoes":
    first = True
    for idx, opt_id in enumerate(OPTS, 1):
        if opt_id not in OPT_MAP:
            print(f"[WARN] '{opt_id}' desconhecida")
            continue
        script = OPT_MAP[opt_id]
        if not os.path.isfile(script):
            print(f"[WARN] {script} não encontrado")
            continue
        print(f"\n[{idx}/{len(OPTS)}] {opt_id} - {OPT_NAMES[opt_id]}")
        args = build_args(script, opt_id, OPT_NAMES[opt_id], ROOT_PATH, DATA, DATA_PATH)
        if first:
            args += ["--reset_csv"]
            first = False
        rc = run_script(args, opt_id)
        if rc != 0:
            errors.append(opt_id)

elif MODO == "bateria":
    for midx, (modelo, llm_dir) in enumerate(zip(BATERIA_MODELOS, BATERIA_LLM_DIRS)):
        script = "run.py"
        if not os.path.isfile(script):
            print(f"[WARN] {script} não encontrado, tentando run_base.py")
            script = "run_base.py"
        if not os.path.isfile(script):
            print("[ERRO] Nenhum script run encontrado")
            break
        for didx, ds in enumerate(BATERIA_DATASETS):
            label = f"{modelo}_{ds['nome']}"
            print(f"\n[{midx+1}/{len(BATERIA_MODELOS)}][{didx+1}/{len(BATERIA_DATASETS)}] {label}")
            args = [
                script,
                "--task_name", BATERIA_TASK, "--is_training", "1",
                "--model", modelo,
                "--root_path", ds["root"],
                "--data_path", ds["path"],
                "--data", ds["type"],
                "--model_id", f"{ds['nome']}_672_96",
                "--llm_ckp_dir", llm_dir,
                "--seq_len", str(SEQ_LEN),
                "--label_len", str(LABEL_LEN),
                "--token_len", str(TOKEN_LEN),
                "--batch_size", str(BATCH_SIZE),
                "--train_epochs", str(EPOCHS),
                "--learning_rate", ds["lr"],
                "--use_amp",
                "--cosine", "--tmax", "10",
                "--des", "Exp",
                "--drop_last",
            ]
            if SMOKE_TEST:
                args += ["--train_epochs=1", "--smoke_test=1"]
            rc = run_script(args, label)
            if rc != 0:
                errors.append(label)

if errors:
    print(f"\n[AVISO] Falhas: {', '.join(errors)}")
else:
    print("\n[Sucesso] Todos concluídos")

---
## 6. Resumo dos resultados

In [ ]:
csv_path = "benchmark_results.csv"
if os.path.isfile(csv_path):
    df = pd.read_csv(csv_path)
    cols = [c for c in ["opt_id", "opt_name", "total_time_s", "test_mse", "test_mae", "peak_vram_gb"] if c in df.columns]
    display(df[cols].to_string(index=False))
else:
    print("benchmark_results.csv não encontrado.")

---
## 7. Baixar resultados para máquina local

In [ ]:
import zipfile
out = "resultados_kaggle.zip"
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ["benchmark_results.csv", "logs/"]:
        p = Path(f)
        if p.is_file():
            zf.write(p)
        elif p.is_dir():
            for sub in p.rglob("*"):
                if sub.is_file():
                    zf.write(sub)
print(f"Baixe o arquivo: {out}")
from IPython.display import FileLink
FileLink(out)